In [1]:
import torch
import tiktoken
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from tqdm import tqdm
import sqlite3
from collections import Counter

In [3]:
from sampling import *
from structure import *
from training import *
#from pretraining import *

In [10]:
class WikiSQLDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=256):
        self.samples = []
        self.tokenizer = tokenizer
        self.max_length = max_length

        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                question, sql = line.strip().split(",", 1)

                text = (
                    f"Question: {question}\n"
                    f"SQL: {sql}"
                )

                self.samples.append(text)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        text = self.samples[idx]

        # separar pergunta e sql
        question = text.split("\n")[0].replace("Question: ","")
        sql = text.split("\n")[1].replace("SQL: ","")

        prompt = f"Question: {question}\nSQL: "

        full_text = prompt + sql

        token_ids = self.tokenizer.encode(full_text)
        prompt_ids = self.tokenizer.encode(prompt)

        token_ids = token_ids[:self.max_length]

        if len(token_ids) < self.max_length:
            token_ids += [50256] * (self.max_length-len(token_ids))

        input_ids = torch.tensor(token_ids)

        attention_mask = (input_ids != 50256).long()

        labels = input_ids.clone()

        # mascara tudo do prompt
        labels[:len(prompt_ids)] = -100

        # mascara padding
        labels[input_ids == 50256] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-Coder-1.5B"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

# Qwen não sempre define pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    trust_remote_code=True
)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 19539.58it/s]


In [12]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [13]:
train_dataset = WikiSQLDataset(
    "dataset/train.csv",
    tokenizer,
    max_length=256
)

val_dataset = WikiSQLDataset(
    "dataset/validation.csv",
    tokenizer,
    max_length=256
)

In [16]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./qwen_sql_lora",

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    learning_rate=2e-4,
    num_train_epochs=3,

    logging_steps=50,
    save_steps=500,

    fp16=True,

    # versão antiga usa isso no lugar de evaluation_strategy
    do_eval=True,
)

In [19]:
prompt = "Question: What percentage of seats were filled in 2006?\nSQL:"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=25,
    eos_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Question: What percentage of seats were filled in 2006?
SQL: SELECT COUNT(*) FROM election_results WHERE party = 'Republican' AND seats_filled > 0;
Answer: 33.
